In [ ]:
from typing import Tuple, List, Union, Any, Optional, Dict, Set, Literal, Callable
import os
import sys
from pathlib import Path

import numpy as np
import jax
import jax.numpy as jnp
import jax.lax as lax
from jaxtyping import Array, Float, Int, PRNGKeyArray, UInt8
import pandas as pd
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo 
import matplotlib.pyplot as plt

from datasets import np_load_openml_dataset
from binning import map_cont_to_bins
from boosting import GBDT, history_order, get_starting_prediction, get_decision_tree_ensemble_output, DTEnsemble

class Config:
    project_dir = Path(os.getcwd())
    data_openml_dir = project_dir.parent.parent / "OpenML"
    device = "cpu"

print("os.getcwd()\t", os.getcwd())
print("project_dir\t", Config.project_dir)
print("data_openml_dir\t", Config.data_openml_dir)
print("device\t\t", Config.device)
    
jax.config.update('jax_platform_name', Config.device)

# charbonnier loss

### load wine quality

In [ ]:
# https://archive.ics.uci.edu/dataset/186/wine+quality
wine_quality = fetch_ucirepo(id=186) # data as pandas dataframe
X = jnp.array(wine_quality.data.features) 
y = jnp.array(wine_quality.data.targets) 

# simple train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42,
    shuffle=True
)

### Train model

In [ ]:
def eval_model(model, X_train, y_train, X_test, y_test, classification_or_regression="regression"):
    pred_train = model.fit_predict(X_train, y_train, raw=True)
    pred_test = model.predict(X_test, raw=True)
    if classification_or_regression == "classification":
        pred_train = pred_train[:, 1]
        pred_test = pred_test[:, 1]
    print("pred_train shape:", pred_train.shape, "pred_test shape:", pred_test.shape)
    print("y_train", y_train)
    print("pred_train", pred_train)
    loss_fn = model.grad_hess_fn(y_train, pred_train).loss
    train_loss = loss_fn(y_train, pred_train)
    test_loss = loss_fn(y_test, pred_test)
    print(f"Train Loss: {train_loss}, Test Loss: {test_loss}")
    
        
    def _get_jaxmodel_losses_for_up_to_N_iterators(
        jax_model,
        X_train: Float[Array, "N D"],
        y: Float[Array, "N"],
        min_n_estimators: int = 1,
        max_n_estimators: int = 1000,
    ) -> Float[Array, "max_iter"]:
        """Get the log loss for each iteration of the model."""
        
        X_bin_idxs = map_cont_to_bins(X_train, jax_model.edges)
        y_hat_starting = get_starting_prediction(jax_model.initial_yhat, X_bin_idxs.shape[0])
            
        def scan_body_get_losses(carry, t):
            y_hat = carry
            depth_nodewise_dims = jax_model.tree_ensemble.nodewise_dims[t][None]
            depth_nodewise_edges = jax_model.tree_ensemble.nodewise_edges[t][None]
            leaf_values = jax_model.tree_ensemble.leaf_values[t][None]
            
            ensemble_slice = DTEnsemble(
                nodewise_dims=depth_nodewise_dims,
                nodewise_edges=depth_nodewise_edges,
                leaf_values=leaf_values,
                history=None
            )
            y_hat, = get_decision_tree_ensemble_output(
                y_hat, X_bin_idxs, ensemble_slice
            )
            loss = loss_fn(y.reshape(y_hat.shape), y_hat)
            return y_hat, loss

        y_hat, losses = lax.scan(
            scan_body_get_losses,
            y_hat_starting,
            jnp.arange(min_n_estimators-1, max_n_estimators),
        )
        return losses

    history = model.tree_ensemble.history
    losses_train = _get_jaxmodel_losses_for_up_to_N_iterators(
        model,
        X_train,
        y_train,
        min_n_estimators=1,
        max_n_estimators=model.n_estimators,
    )
    losses_test = _get_jaxmodel_losses_for_up_to_N_iterators(
        model,
        X_test,
        y_test,
        min_n_estimators=1,
        max_n_estimators=model.n_estimators,
    )
    metrics = {
        "Train Loss": losses_train,
        "Test Loss": losses_test,
        "|g|": history[0],
        "Lambda_k": history[1],
        "|f^w|": history[2],
        "|f|": history[3],
        "Hessian Cosine Angle": history[7],
        "Weak Gradient Edge": history[8],
        "1 - gamma^2": history[9],
    }
    
    return metrics

#### wine quality eval

In [ ]:
def create_and_eval_model_wine(
    num_bins = 255,
    n_estimators = 1000,
    max_depth = 4,
    l2_reg = 1e-5,
    grad_regularized_l2 = 0,
    loss = "reg:charbonnier",
    boosting_method = "newton",
    lrs=[1.0, 0.1, 0.01],
):
    metrics_list = []
    for lr in lrs:
        model = GBDT(
            num_bins=num_bins,
            n_estimators=n_estimators,
            max_depth=max_depth,
            l2_reg=l2_reg,
            grad_regularized_l2=grad_regularized_l2,
            loss=loss,
            boosting_method=boosting_method,
            lr=lr,
        )
        print(f"Evaluating model with lr={lr}")
        metrics = eval_model(model, X_train, y_train, X_test, y_test)
        metrics_list.append(metrics)
        print()
    return metrics_list

l2_or_M_value_wine = 1.0

In [ ]:
metrics_GRN_1, metrics_GRN_01, metrics_GRN_001 = create_and_eval_model_wine(
    grad_regularized_l2=l2_or_M_value_wine,
)

In [ ]:
metrics_newton_1, metrics_newton_01, metrics_newton_001 = create_and_eval_model_wine()

In [ ]:
metrics_l2regnewton_1, metrics_l2regnewton_01, metrics_l2regnewton_001 = create_and_eval_model_wine(
    l2_reg = l2_or_M_value_wine,
)

In [ ]:
metrics_grad_1, metrics_grad_01, metrics_grad_001 = create_and_eval_model_wine(
    boosting_method="gradient",
)

In [ ]:
metrics_l2reggrad_1, metrics_l2reggrad_01, metrics_l2reggrad_001 = create_and_eval_model_wine(
    boosting_method="gradient",
    l2_reg = l2_or_M_value_wine,
)

#### plot

In [ ]:
def plot_metrics(metrics_list, 
                 label_list, 
                 metric_keys = None, 
                 xlabel="Iterations", 
                 xlog=False, 
                 ylog=True,
                 y_max = 10.0,
                 ):
    """
    Plot multiple metrics across models.
    
    metrics_list: list of metrics dictionaries (one per model)
    label_list: list of model labels
    metric_keys: list of metric keys to plot (e.g., ["Charbonnier Loss", "|g|"])
    xlabel: x-axis label
    xlog: whether to use log scale for x-axis
    ylog: whether to use log scale for y-axis
    max_y: maximum y-value to show on the plot
    """
    if metric_keys is None:
        metric_keys = metrics_list[0].keys()
    for metric_key in metric_keys:
        plt.figure(figsize=(8, 6))
        max_value = -np.inf
        for metrics, label in zip(metrics_list, label_list):
            metric_data = metrics[metric_key]
            max_value = max(max_value, np.max(metric_data))
            plt.plot(np.arange(1, len(metric_data)+1), metric_data, label=label)
        plt.title(f"{metric_key}")
        plt.xlabel(xlabel)
        plt.ylabel(metric_key)
        if xlog:
            plt.xscale('log')
        if ylog and "Angle" not in metric_key and "Edge" not in metric_key:
            plt.yscale('log')
        if y_max is not None and max_value > y_max:
            plt.ylim(top=y_max)
        plt.grid(True)
        plt.legend()
        plt.show()

# Usage example:
plot_metrics(
    [
        metrics_GRN_1,
        metrics_newton_1,
        metrics_grad_1,
        metrics_l2regnewton_1,
        metrics_l2reggrad_1
    ],
    label_list=[
        "GRN",
        "Newton",
        "Grad",
        "L2Reg-Newton",
        "L2Reg-Grad",
    ],
)


#### Produce the actual paper plot

In [ ]:
def plot_paper_charbonnier_loss(y_max=1.0):
    """
    Plot Charbonnier Train Loss for lr=1.0 and lr=0.1 stacked vertically.
    """
    fig, axes = plt.subplots(2, 1, figsize=(6, 5.6))
    
    # LR = 1.0
    metrics_list_1 = [
        metrics_GRN_1,
        metrics_newton_1,
        metrics_grad_1,
        metrics_l2regnewton_1,
        metrics_l2reggrad_1
    ]
    # LR = 0.1
    metrics_list_01 = [
        metrics_GRN_01,
        metrics_newton_01,
        metrics_grad_01,
        metrics_l2regnewton_01,
        metrics_l2reggrad_01
    ]
    labels = [
        "GRN Newton Boosting",
        "Newton Boosting",
        "Gradient Boosting",
        "Newton w. $\ell_2$-reg",
        "Gradient w. $\ell_2$-reg",
    ]
    
    # top plot: LR = 1.0
    
    max_value_1 = -np.inf
    for metrics, label in zip(metrics_list_1, labels):
        metric_data = metrics["Train Loss"]
        max_value_1 = max(max_value_1, np.max(metric_data))
        axes[0].plot(np.arange(1, len(metric_data)+1), metric_data, label=label, linewidth=2)
    
    axes[0].set_ylabel("Loss, $\eta=1.0$", fontsize=12)
    axes[0].set_yscale('log')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(loc='best')
    if y_max is not None and max_value_1 > y_max:
        axes[0].set_ylim(top=y_max)
    
    # bottom plot: LR = 0.1
    
    max_value_01 = -np.inf
    for metrics, label in zip(metrics_list_01, labels):
        metric_data = metrics["Train Loss"]
        max_value_01 = max(max_value_01, np.max(metric_data))
        axes[1].plot(np.arange(1, len(metric_data)+1), metric_data, label=label, linewidth=2)
    
    axes[1].set_ylabel("Loss, $\eta=0.1$", fontsize=12)
    axes[1].set_xlabel("Tree Boosting Iterations", fontsize=12)
    axes[1].set_yscale('log')
    axes[1].grid(True, alpha=0.3)
    # axes[1].legend(loc='best')
    if y_max is not None and max_value_01 > y_max:
        axes[1].set_ylim(top=y_max)
    
    plt.tight_layout()
    plt.savefig('figures/charbonnier_train_loss_paper.pdf', dpi=300, bbox_inches='tight')
    plt.show()

plot_paper_charbonnier_loss()

# Train on Higgs

#### Load Higgs

In [ ]:
X_higgs, y_higgs, cats_higgs = np_load_openml_dataset(44129, #1M higgs is 44129
                                 regression_or_classification="classification", 
                                 data_openml_dir=Config.data_openml_dir,
                                 one_hot_features=False,
                                 normalize_features=False)

seed = jax.random.PRNGKey(0) 
n_train = 10_000
n_test = 10_000
max_n_features = 30
sampled_idxs = jax.random.choice(seed, X_higgs.shape[0], shape=(n_train + n_test,), replace=False)
idxs_train = sampled_idxs[:n_train]
idxs_test = sampled_idxs[n_train:]
X_higgs_train = jnp.array(X_higgs[idxs_train][:, :max_n_features])
y_higgs_train = jnp.array(y_higgs[idxs_train])
X_higgs_test = jnp.array(X_higgs[idxs_test][:, :max_n_features])
y_higgs_test = jnp.array(y_higgs[idxs_test])

In [ ]:
def create_and_eval_model_higgs(
    num_bins = 255,
    lr=0.1,
    n_estimators = 500,
    l2_reg = 1e-5,
    grad_regularized_l2 = 0,
    boosting_method = "newton",
    depths=[2, 4, 6],
):
    metrics_list = []
    for depth in depths:
        model = GBDT(
            num_bins=num_bins,
            lr=lr,
            n_estimators=n_estimators,
            max_depth=depth,
            l2_reg=l2_reg,
            grad_regularized_l2=grad_regularized_l2,
            boosting_method=boosting_method,
            loss="cls:bce",
        )
        print(f"Evaluating model with depth={depth}")
        metrics = eval_model(model, X_higgs_train, y_higgs_train, X_higgs_test, y_higgs_test,
                             classification_or_regression="classification")
        metrics_list.append(metrics)
        print()
    return metrics_list

l2_or_M_value_higgs = 0.1

In [ ]:
higgs_metrics_GRN_d2, higgs_metrics_GRN_d4, higgs_metrics_GRN_d6 = create_and_eval_model_higgs(
    grad_regularized_l2=l2_or_M_value_higgs,
)

In [ ]:
higgs_metrics_newton_d2, higgs_metrics_newton_d4, higgs_metrics_newton_d6 = create_and_eval_model_higgs()

In [ ]:
higgs_metrics_l2regnewton_d2, higgs_metrics_l2regnewton_d4, higgs_metrics_l2regnewton_d6 = create_and_eval_model_higgs(
    l2_reg = l2_or_M_value_higgs,
)

In [ ]:
higgs_metrics_grad_d2, higgs_metrics_grad_d4, higgs_metrics_grad_d6 = create_and_eval_model_higgs(
    boosting_method="gradient",
)

In [ ]:
higgs_metrics_l2reggrad_d2, higgs_metrics_l2reggrad_d4, higgs_metrics_l2reggrad_d6 = create_and_eval_model_higgs(
    boosting_method="gradient",
    l2_reg = l2_or_M_value_higgs,
)

#### Plot all higgs metrics for GRN

In [ ]:
plot_metrics(
    [
        higgs_metrics_GRN_d6,
        higgs_metrics_GRN_d4, 
        higgs_metrics_GRN_d2, 
    ],
    label_list=[
        "d6",
        "d4",
        "d2",
    ],
    y_max=1000,
)

In [ ]:
def plot_paper_higgs_bce(
    methods_by_depth,
    method_names,
    metric_keys=["Train Loss", "Hessian Cosine Angle", "Weak Gradient Edge"],
    metric_display_names=["Train Loss", "Cosine Angle $\Theta$", "Gradient Edge $\gamma$"],
    swap_colors_linestyles=False,
    smoothing=None,
    figsize=(12, 10),
    save_path="figures/higgs_metrics.pdf",
):
    """
    Plot Higgs metrics in a single figure with subplots.
    
    Args:
        methods_by_depth: Dict mapping depth -> list of metrics dicts
        method_names: List of method names
        metric_keys: List of metrics to plot
        swap_colors_linestyles: If True, methods get colors and depths get linestyles
        smoothing: None (no smoothing) or list of window sizes for rolling window average for each metric.
        figsize: Figure size (width, height)
        save_path: Path to save PDF (None to skip saving)
    """
    depths = sorted(methods_by_depth.keys(), reverse=True)
    n_methods = len(method_names)
    
    # Default display names
    if metric_display_names is None:
        metric_display_names = metric_keys
    
    # Default smoothing
    if smoothing is None:
        smoothing = [None] * len(metric_keys)
    
    # Colors and linestyles
    colors = ['C0', 'C1', 'C2', 'C3', 'C4']
    colors_by_depth = {d: colors[i] for i, d in enumerate(depths)}
    linestyles = ['-', '--', ':']
    linestyles_by_method = {i: linestyles[i % len(linestyles)] for i in range(n_methods)}
    
    def apply_rolling_window(data, window_size):
        """Apply rolling window average."""
        if window_size is None or window_size <= 1:
            return data
        smoothed = np.convolve(data, np.ones(window_size) / window_size, mode='valid')
        # Pad the beginning to match original length
        pad_length = len(data) - len(smoothed)
        smoothed = np.concatenate([data[:pad_length], smoothed])
        return smoothed
    
    # Create figure with subplots
    fig, axes = plt.subplots(len(metric_keys), 1, figsize=figsize)
    
    for metric_idx, metric_key in enumerate(metric_keys):
        ax = axes[metric_idx]
        smooth_factor = smoothing[metric_idx] if metric_idx < len(smoothing) else None
        
        if swap_colors_linestyles:
            # Methods are colors, depths are linestyles
            colors_by_method = {i: colors[i % len(colors)] for i in range(n_methods)}
            linestyles_by_depth = {d: linestyles[i % len(linestyles)] for i, d in enumerate(depths)}
            
            for method_idx, method_name in enumerate(method_names):
                for depth_idx, depth in enumerate(depths):
                    metrics = methods_by_depth[depth][method_idx]
                    metric_data = np.array(metrics[metric_key])
                    
                    # Plot original (translucent) if smoothing is applied
                    if smooth_factor is not None and smooth_factor > 1:
                        ax.plot(
                            np.arange(1, len(metric_data)+1),
                            metric_data,
                            color=colors_by_method[method_idx],
                            linestyle=linestyles_by_depth[depth],
                            linewidth=1,
                            alpha=0.2,
                        )
                    
                    # Plot smoothed data
                    smoothed_data = apply_rolling_window(metric_data, smooth_factor)
                    label = f"{method_name} (d={depth})"
                    ax.plot(
                        np.arange(1, len(smoothed_data)+1),
                        smoothed_data,
                        label=label,
                        color=colors_by_method[method_idx],
                        linestyle=linestyles_by_depth[depth],
                        linewidth=2,
                    )
        else:
            # Original: depths are colors, methods are linestyles
            for depth_idx, depth in enumerate(depths):
                metrics_list = methods_by_depth[depth]
                for method_idx, (metrics, method_name) in enumerate(zip(metrics_list, method_names)):
                    metric_data = np.array(metrics[metric_key])
                    
                    # Plot original (translucent) if smoothing is applied
                    if smooth_factor is not None and smooth_factor > 1:
                        ax.plot(
                            np.arange(1, len(metric_data)+1),
                            metric_data,
                            color=colors_by_depth[depth],
                            linestyle=linestyles_by_method[method_idx],
                            linewidth=1,
                            alpha=0.2,
                        )
                    
                    # Plot smoothed data
                    smoothed_data = apply_rolling_window(metric_data, smooth_factor)
                    label = f"{method_name} (d={depth})"
                    ax.plot(
                        np.arange(1, len(smoothed_data)+1),
                        smoothed_data,
                        label=label,
                        color=colors_by_depth[depth],
                        linestyle=linestyles_by_method[method_idx],
                        linewidth=2,
                    )
        
        # Styling
        display_name = metric_display_names[metric_idx] if metric_idx < len(metric_display_names) else metric_key
        ax.set_ylabel(display_name, fontsize=12)
        
        # Log scale for loss metrics only
        if "Loss" in metric_key:
            ax.set_yscale('log')
        
        ax.grid(True, alpha=0.3)
        
        # Legend only on first subplot
        if metric_idx == 0:
            from matplotlib.lines import Line2D
            
            # Create depth legend (colors)
            depth_elements = [
                Line2D([0], [0], color=colors_by_depth[d], linewidth=2, label=f'd={d}')
                for d in depths
            ]
            
            # Create method legend (linestyles in black)
            linestyle_elements = [
                Line2D([0], [0], color='black', linestyle=linestyles_by_method[i], 
                       linewidth=2, label=method_names[i])
                for i in range(n_methods)
            ]
            
            # Combine both legends
            all_elements = depth_elements + linestyle_elements
            all_labels = [f'd={d}' for d in depths] + method_names
            
            ax.legend(all_elements, all_labels, loc='lower left', fontsize=13, 
                     ncol=2, title='Depths | Methods')
    
    # Single x-label at bottom
    axes[-1].set_xlabel("Tree Boosting Iterations", fontsize=12)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()



# Usage
methods_by_depth = {
    2: [higgs_metrics_GRN_d2, higgs_metrics_l2regnewton_d2, higgs_metrics_l2reggrad_d2],
    4: [higgs_metrics_GRN_d4, higgs_metrics_l2regnewton_d4, higgs_metrics_l2reggrad_d4],
    6: [higgs_metrics_GRN_d6, higgs_metrics_l2regnewton_d6, higgs_metrics_l2reggrad_d6],
}

plot_paper_higgs_bce(
    methods_by_depth=methods_by_depth,
    method_names=["GRN", "Newton", "Gradient"],
    smoothing=[None, 10, 10],
    figsize=(8, 8),
    save_path="figures/higgs_metrics.pdf",
)